In [10]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
train=pd.read_csv("/content/drive/MyDrive/calcofi/train.csv")
test=pd.read_csv("/content/drive/MyDrive/calcofi/test.csv")
sub=pd.read_csv("/content/drive/MyDrive/calcofi/sample_submission.csv")

### Googleドライブからデータを読み込む設定

Googleドライブのマウントが完了したら、以下のセルのパスをご自身のGoogleドライブ内のファイルパス（例: `/content/drive/MyDrive/...`）に書き換えて実行してください。

※ 以下はマイドライブ直下の `calcofi` フォルダにファイルを配置した場合の記述例です。

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1454 entries, 0 to 1453
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1454 non-null   int64  
 1   Lat_Dec            1454 non-null   float64
 2   Lon_Dec            1454 non-null   float64
 3   NO2uM              1454 non-null   float64
 4   NO3uM              1454 non-null   float64
 5   NH3uM              1454 non-null   float64
 6   R_TEMP             1454 non-null   float64
 7   R_Depth            1454 non-null   int64  
 8   R_Sal              1454 non-null   float64
 9   R_DYNHT            1454 non-null   float64
 10  R_Nuts             1454 non-null   float64
 11  R_Oxy_micromol.Kg  1454 non-null   float64
 12  Unnamed: 12        0 non-null      float64
 13  PO4uM              1454 non-null   float64
 14  SiO3uM             1454 non-null   float64
 15  TA1.x              1454 non-null   float64
 16  Salinity1          1454 

In [14]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load data
# train = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
# test = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

# Find common columns between train and test (excluding "DIC" from train)
common_columns = train.drop(columns=["DIC"]).columns.intersection(test.columns)

# Select the common columns for both train and test
X = train[common_columns]
y = train["DIC"]
test = test[common_columns]

# Replace NaNs with the column mean
# X.fillna(X.mean(), inplace=True)
# test.fillna(test.mean(), inplace=True)
X.fillna(X.mean())
test.fillna(test.mean())

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

# Split the training data as before
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [15]:
from torch.utils.data import Dataset, DataLoader

class OceanChemistryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = OceanChemistryDataset(X_train, y_train)
val_dataset = OceanChemistryDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [16]:
class MLPModel(nn.Module):
    def __init__(self, input_size):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 1)  # Output layer for regression

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# Initialize the model
model = MLPModel(input_size=X_train.shape[1])
print(X_train.shape)


(1163, 15)


In [17]:
import torch.optim as optim

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training function
def train_model(model, train_loader, val_loader, epochs=5000):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs.squeeze(), y_batch)
                val_loss += loss.item()

        if epoch % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {running_loss/len(train_loader)}, Validation Loss: {val_loss/len(val_loader)}")

# Train the model
train_model(model, train_loader, val_loader, epochs=5000)


Epoch 1/5000, Train Loss: 4627006.552631579, Validation Loss: 4610291.7
Epoch 101/5000, Train Loss: 14169.531147203947, Validation Loss: 16677.0158203125
Epoch 201/5000, Train Loss: 2441.835545590049, Validation Loss: 3344.195068359375
Epoch 301/5000, Train Loss: 822.4770636307566, Validation Loss: 1062.715869140625
Epoch 401/5000, Train Loss: 427.26895302220396, Validation Loss: 604.6127258300781
Epoch 501/5000, Train Loss: 228.14196456106086, Validation Loss: 287.1433074951172
Epoch 601/5000, Train Loss: 150.11999391254625, Validation Loss: 188.04925842285155
Epoch 701/5000, Train Loss: 110.62622150621917, Validation Loss: 104.98519439697266
Epoch 801/5000, Train Loss: 81.21063553659539, Validation Loss: 76.4611572265625
Epoch 901/5000, Train Loss: 58.127286409076895, Validation Loss: 91.17361297607422
Epoch 1001/5000, Train Loss: 45.81075557909514, Validation Loss: 59.65341300964356
Epoch 1101/5000, Train Loss: 43.82068694265265, Validation Loss: 63.63862228393555
Epoch 1201/5000, T

In [20]:
# Convert the test set into a torch tensor
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

# Set the model to evaluation mode
model.eval()

# Make predictions
with torch.no_grad():
    predictions = model(test_tensor).squeeze().numpy()

# Prepare submission
submission = pd.DataFrame({"id": range(1455, 1455 + len(predictions)), "DIC": predictions})
submission.to_csv("submission.csv", index=False)


In [19]:
# score: 6.08783
# name: 杉浦孔明
# id: 123456

- 5000エポックではなく、Early Stopping
- 標準化に用いる平均値と標準偏差は、全体のそれではなく訓練データのものだけにする